# Generate VLM Responses for Eagle3 Training

This notebook generates responses from the target VLM model (Qwen3-VL-30B-A3B) for all dataset images.

**Prerequisites:** Run `prepare_datasets.ipynb` first!

**Estimated time:** 8-12 hours (A100 GPU)
**GPU memory:** ~35GB
**Output:** Conversations in ShareGPT format

## Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Change to AngelSlim directory
%cd /content/AngelSlim

In [ ]:
# Install dependencies
!pip install -q transformers>=4.37.0 accelerate torch pillow tqdm

In [ ]:
# Add to path
import sys
sys.path.insert(0, '/content/AngelSlim')

CONFIG = {
    # Model - ВАЖНО: нужна A100 80GB для 30B!
    'model_name': 'Qwen/Qwen3-VL-30B-A3B',  # Requires A100 80GB
    # Для A100 40GB используйте: 'Qwen/Qwen3-VL-4B'
    # Для быстрых тестов: 'Qwen/Qwen3-VL-2B'
    'torch_dtype': 'bfloat16',
    
    # Datasets
    'datasets': [
        {
            'name': 'sharegpt4v_en',
            'input': '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/sharegpt4v_en/data_raw.jsonl',
            'output': '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/sharegpt4v_en/data_generated.jsonl',
            'images_dir': '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/sharegpt4v_en/images',
        },
        {
            'name': 'm3it_zh',
            'input': '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/m3it_zh/data_raw.jsonl',
            'output': '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/m3it_zh/data_generated.jsonl',
            'images_dir': '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/m3it_zh/images',
        },
    ],
    
    # Generation settings
    'batch_size': 1,  # Keep at 1 for 30B model
    'max_new_tokens': 512,
    'temperature': 0.7,
    'save_interval': 100,
}

print("Configuration:")
for key, value in CONFIG.items():
    if key != 'datasets':
        print(f"  {key}: {value}")

In [ ]:
CONFIG = {
    # Model
    'model_name': 'Qwen/Qwen3-VL-30B-A3B',  # Change to Qwen3-VL-2B for quick test
    'torch_dtype': 'bfloat16',
    
    # Datasets
    'datasets': [
        {
            'name': 'sharegpt4v_en',
            'input': '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/sharegpt4v_en/data_raw.jsonl',
            'output': '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/sharegpt4v_en/data_generated.jsonl',
            'images_dir': '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/sharegpt4v_en/images',
        },
        {
            'name': 'm3it_zh',
            'input': '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/m3it_zh/data_raw.jsonl',
            'output': '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/m3it_zh/data_generated.jsonl',
            'images_dir': '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/m3it_zh/images',
        },
    ],
    
    # Generation settings
    'batch_size': 1,  # Keep at 1 for 30B model
    'max_new_tokens': 512,
    'temperature': 0.7,
    'save_interval': 100,
}

print("Configuration:")
for key, value in CONFIG.items():
    if key != 'datasets':
        print(f"  {key}: {value}")

## Download and Cache Model

In [ ]:
from colab_code.utils import get_cache_manager

cache_manager = get_cache_manager()

# Download and cache model
model_cache_name = CONFIG['model_name'].replace('/', '_')

print(f"Downloading and caching {CONFIG['model_name']}...")
print("This may take 30-60 minutes for 30B model")

model_path = cache_manager.download_and_cache_model(
    model_name_or_path=CONFIG['model_name'],
    cache_name=model_cache_name,
)

print(f"✅ Model cached at: {model_path}")

## Generate Responses for English Dataset

In [ ]:
import subprocess

en_dataset = CONFIG['datasets'][0]

print("="*60)
print(f"Generating responses for {en_dataset['name']}")
print("="*60)

cmd = [
    'python', 'tools/generate_vlm_responses.py',
    '--model_name_or_path', model_path,
    '--input_data_path', en_dataset['input'],
    '--output_data_path', en_dataset['output'],
    '--images_dir', en_dataset['images_dir'],
    '--batch_size', str(CONFIG['batch_size']),
    '--max_new_tokens', str(CONFIG['max_new_tokens']),
    '--temperature', str(CONFIG['temperature']),
    '--torch_dtype', CONFIG['torch_dtype'],
    '--save_interval', str(CONFIG['save_interval']),
]

result = subprocess.run(cmd)

if result.returncode == 0:
    print(f"\n✅ English dataset responses generated successfully")
else:
    print(f"\n❌ Generation failed with code {result.returncode}")

## Generate Responses for Chinese Dataset

In [ ]:
zh_dataset = CONFIG['datasets'][1]

print("="*60)
print(f"Generating responses for {zh_dataset['name']}")
print("="*60)

cmd = [
    'python', 'tools/generate_vlm_responses.py',
    '--model_name_or_path', model_path,
    '--input_data_path', zh_dataset['input'],
    '--output_data_path', zh_dataset['output'],
    '--images_dir', zh_dataset['images_dir'],
    '--batch_size', str(CONFIG['batch_size']),
    '--max_new_tokens', str(CONFIG['max_new_tokens']),
    '--temperature', str(CONFIG['temperature']),
    '--torch_dtype', CONFIG['torch_dtype'],
    '--save_interval', str(CONFIG['save_interval']),
]

result = subprocess.run(cmd)

if result.returncode == 0:
    print(f"\n✅ Chinese dataset responses generated successfully")
else:
    print(f"\n❌ Generation failed with code {result.returncode}")

## Validation and Summary

In [ ]:
import json

def count_samples(jsonl_file):
    """Count samples in JSONL file."""
    try:
        with open(jsonl_file, 'r') as f:
            return sum(1 for _ in f)
    except FileNotFoundError:
        return 0

# Count generated samples
en_count = count_samples(en_dataset['output'])
zh_count = count_samples(zh_dataset['output'])
total_count = en_count + zh_count

print("="*60)
print("RESPONSE GENERATION COMPLETE")
print("="*60)

print(f"\n📊 Summary:")
print(f"  English responses: {en_count}")
print(f"  Chinese responses: {zh_count}")
print(f"  Total responses: {total_count}")
print(f"  Language distribution: {en_count/total_count*100:.1f}% EN / {zh_count/total_count*100:.1f}% ZH")

print(f"\n📁 Saved to:")
print(f"  English: {en_dataset['output']}")
print(f"  Chinese: {zh_dataset['output']}")

print(f"\n✅ Next step:")
print(f"  Run generate_hidden_states.ipynb to extract hidden states")

## Optional: View Generated Samples

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

def show_generated_sample(output_file, images_dir, title):
    """Show a random generated conversation."""
    with open(output_file, 'r') as f:
        lines = f.readlines()
        sample = json.loads(random.choice(lines))
    
    # Extract conversation
    conversations = sample['conversations']
    question = conversations[0]['value'].replace('<image>', '')
    answer = conversations[1]['value']
    
    # Load image
    img_path = f"{images_dir.rsplit('/', 1)[0]}/{sample['img_path'].lstrip('./')}"
    img = Image.open(img_path)
    
    # Display
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.title(f"{title}\n\nQ: {question}\n\nA: {answer[:200]}..." if len(answer) > 200 else f"{title}\n\nQ: {question}\n\nA: {answer}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Show English sample
show_generated_sample(
    en_dataset['output'],
    en_dataset['images_dir'],
    "English Sample with Generated Response"
)

# Show Chinese sample
show_generated_sample(
    zh_dataset['output'],
    zh_dataset['images_dir'],
    "Chinese Sample with Generated Response"
)